# 📊 EDA Checklist — Notebook Modelo
> Use este notebook como ponto de partida para QUALQUER dataset.
> Siga os passos em ordem antes de fazer qualquer análise ou gráfico.

---

## ✅ PASSO 1 — Importar bibliotecas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# Configurações visuais
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ Bibliotecas carregadas com sucesso!')

## ✅ PASSO 2 — Carregar o dataset
> ⚠️ Altere o caminho do arquivo para o seu dataset

In [ ]:
# 👇 Altere o caminho aqui
df = pd.read_csv('survey.csv')

print(f'✅ Dataset carregado!')
print(f'   Linhas: {df.shape[0]}')
print(f'   Colunas: {df.shape[1]}')

## ✅ PASSO 3 — Visão geral do dataset

In [ ]:
# Ver as primeiras 5 linhas
df.head()

In [ ]:
# Ver as últimas 5 linhas
df.tail()

In [ ]:
# Tipos de dados e valores nulos
df.info()

## ✅ PASSO 4 — Verificar valores nulos
> Colunas com muitos nulos podem precisar de tratamento antes da análise

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (df.isnull().sum() / len(df) * 100).round(1)

resumo_nulos = pd.DataFrame({
    'Nulos (qtd)': nulos,
    'Nulos (%)': nulos_pct
})

# Mostrar só colunas com nulos
print('=== Colunas com valores nulos ===')
print(resumo_nulos[resumo_nulos['Nulos (qtd)'] > 0])

## ✅ PASSO 5 — Verificar se os grupos são equilibrados
> ⚠️ CRÍTICO: grupos desequilibrados exigem taxa proporcional, não volume absoluto

> 👇 Altere 'coluna_do_grupo' para a coluna que você quer analisar

In [ ]:
# 👇 Altere aqui para a sua coluna de grupo
coluna_grupo = 'Gender'

print(f'=== Distribuição de {coluna_grupo} ===')
print()

# Volume absoluto
print('Volume absoluto:')
print(df[coluna_grupo].value_counts())
print()

# Percentual
print('Percentual:')
print((df[coluna_grupo].value_counts(normalize=True) * 100).round(1))
print()

# Alerta automático
pct_max = (df[coluna_grupo].value_counts(normalize=True) * 100).max()
if pct_max > 60:
    print(f'⚠️  ATENÇÃO: O grupo maior representa {pct_max:.1f}% do dataset.')
    print('   Use TAXA PROPORCIONAL nas suas análises!')
else:
    print('✅ Grupos relativamente equilibrados. Volume absoluto pode ser usado.')

## ✅ PASSO 6 — Explorar valores únicos de cada coluna
> Aqui você descobre erros de digitação, inconsistências e necessidade de limpeza

In [ ]:
print('=== Valores únicos por coluna ===')
print()

for col in df.columns:
    n_unicos = df[col].nunique()
    print(f'{col}: {n_unicos} valores únicos')
    
    # Se poucos valores únicos, mostra todos
    if n_unicos <= 10:
        print(f'   → {df[col].value_counts().to_dict()}')
    print()

## ✅ PASSO 7 — Resumo estatístico das colunas numéricas

In [ ]:
df.describe().round(1)

## ✅ PASSO 8 — Resumo completo das colunas (tipo + nulos + únicos)

In [ ]:
resumo = pd.DataFrame({
    'Tipo': df.dtypes,
    'Nulos (qtd)': df.isnull().sum(),
    'Nulos (%)': (df.isnull().sum() / len(df) * 100).round(1),
    'Únicos': df.nunique(),
    'Exemplo': df.iloc[0]
})

print('=== Resumo completo das colunas ===')
resumo

## ✅ PASSO 9 — Limpeza de colunas com texto livre
> Use este bloco quando encontrar colunas com muitas variações do mesmo valor

> Exemplo aplicado: coluna Gender do dataset survey.csv

In [ ]:
# 👇 Adapte esta função para a sua coluna
def padronizar_genero(valor):
    v = str(valor).lower().strip()
    
    if v in ['male', 'm', 'man', 'make', 'maile', 'mal',
             'male (cis)', 'cis male', 'male-ish',
             'something kinda male?']:
        return 'Male'
    
    elif v in ['female', 'f', 'woman', 'femail', 'femake',
               'cis female', 'female (trans)', 'trans-female']:
        return 'Female'
    
    else:
        return 'Other'

df['Genero'] = df['Gender'].apply(padronizar_genero)

print('=== Resultado da limpeza ===')
print(df['Genero'].value_counts())
print()
print(f'Total: {df["Genero"].value_counts().sum()} (deve ser igual ao nº de linhas)')

In [ ]:
# 1 gráfico
plt.subplots(1, 1, figsize=(8, 5))

# 2 gráficos lado a lado
plt.subplots(1, 2, figsize=(12, 5))

# 2 gráficos um embaixo do outro
plt.subplots(2, 1, figsize=(8, 10))

# 4 gráficos em grade
plt.subplots(2, 2, figsize=(12, 10))

## ✅ PASSO 10 — Análise: Volume vs Taxa Proporcional
> Sempre gere os dois gráficos para comparar

> 👇 Altere as variáveis abaixo para a sua análise

In [ ]:
# 👇 Altere aqui
coluna_grupo   = 'Genero'      # variável de agrupamento
coluna_alvo    = 'treatment'   # variável que você quer analisar
valor_positivo = 'Yes'         # valor que representa o evento

# --- Volume absoluto ---
df_positivo = df[df[coluna_alvo] == valor_positivo]
volume = df_positivo[coluna_grupo].value_counts()

# --- Taxa proporcional ---
taxa = df.groupby(coluna_grupo)[coluna_alvo].apply(
    lambda x: (x == valor_positivo).sum() / len(x) * 100
).round(1)

print('=== Volume absoluto ===')
print(volume)
print()
print('=== Taxa proporcional (%) ===')
print(taxa)

# --- Gráficos ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

volume.plot(kind='bar', ax=axes[0],
            color=['#2E75B6', '#C55A11', '#7030A0'],
            edgecolor='none')
axes[0].set_title(f'Volume: quem tem {coluna_alvo} = {valor_positivo}')
axes[0].set_xlabel('')
axes[0].set_ylabel('Nº de pessoas')
axes[0].tick_params(axis='x', rotation=0)

taxa.plot(kind='bar', ax=axes[1],
          color=['#C55A11', '#2E75B6', '#7030A0'],
          edgecolor='none')
axes[1].set_title(f'Taxa proporcional (%) por {coluna_grupo}')
axes[1].set_xlabel('')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylim(0, 100)

plt.suptitle('⚠️ Compare sempre os dois gráficos!', fontsize=11, color='gray')
plt.tight_layout()
plt.show()

## ✅ PASSO 11 — Teste Qui-quadrado
> Verifica se a associação entre duas variáveis é estatisticamente significativa
>
> Regra: se p < 0.05 → associação significativa ✅

In [ ]:
# 👇 Altere as colunas aqui
col1 = 'Genero'
col2 = 'treatment'

tabela = pd.crosstab(df[col1], df[col2])
chi2, p, dof, expected = chi2_contingency(tabela)

print(f'=== Teste Qui-quadrado: {col1} vs {col2} ===')
print(f'   chi2 = {chi2:.2f}')
print(f'   p    = {p:.4f}')
print(f'   dof  = {dof}')
print()

if p < 0.05:
    print('✅ Associação SIGNIFICATIVA (p < 0.05)')
    print('   As variáveis estão relacionadas.')
else:
    print('❌ Associação NÃO significativa (p >= 0.05)')
    print('   Não há evidência de relação entre as variáveis.')

print()
print('=== Tabela cruzada (%) ===')
print(pd.crosstab(df[col1], df[col2], normalize='index').round(2) * 100)

---
## 📋 CHECKLIST RÁPIDO

Antes de qualquer análise, confirme:

- [ ] Dataset carregado e shape verificado
- [ ] Valores nulos identificados
- [ ] Grupos equilibrados? (se não → usar taxa proporcional)
- [ ] Valores únicos inspecionados (erros de digitação?)
- [ ] Colunas de texto livre limpas e padronizadas
- [ ] Sempre gerar volume E taxa proporcional
- [ ] Teste qui-quadrado para validar associações

---
*Notebook modelo criado para prática de EDA em datasets de saúde mental corporativa.*